In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import os
from sklearn.preprocessing import OrdinalEncoder
import lightgbm as lgb
import os, sys, gc, time, warnings, pickle, psutil, random
from tqdm import tqdm
from math import ceil
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import math
from statsmodels.tsa.seasonal import STL
from scipy import stats

In [ ]:
def downcast(df):
    cols = df.dtypes.index.tolist()
    types = df.dtypes.values.tolist()
    
    for i, t in enumerate(types):
        col = cols[i]
        if 'int' in str(t):
            c_min = df[col].min()
            c_max = df[col].max()
            if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                df[col] = df[col].astype(np.int8)
            elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                df[col] = df[col].astype(np.int16)
            elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                df[col] = df[col].astype(np.int32)
            else:
                df[col] = df[col].astype(np.int64)

        elif 'float' in str(t):
            c_min = df[col].min()
            c_max = df[col].max()
            # ⚠️ Không dùng float16 vì không lưu được trong Parquet
            if c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                df[col] = df[col].astype(np.float32)
            else:
                df[col] = df[col].astype(np.float64)    
    return df

In [ ]:
p = pd.read_parquet(r"C:\Users\ADMIN\Downloads\train_val.parquet")
# giữ lại các dòng có demand là nan
p = p[p['demand'].isna()]

print(p.shape)


ArrowMemoryError: realloc of size 268435456 failed

In [ ]:
cal = pd.read_parquet(r"C:\Users\ADMIN\Downloads\done_cal.parquet")
# Đọc dữ liệu
grid_df = pd.read_parquet(r"C:\Users\ADMIN\Downloads\train_val.parquet")
# grid_df = grid_df.drop(['date'], axis=1)
grid_df = grid_df.dropna(subset=['demand'])
# print(grid_df.columns)
# print(len(duplicate_id_list))

In [ ]:
grid_df = grid_df.sort_values(['id', 'date'])

In [ ]:
duplicate_id_list = grid_df[grid_df.duplicated('id', keep=False)]['id'].unique()

print(duplicate_id_list[0:10])

['FOODS_1_001_CA_1' 'FOODS_1_001_CA_2' 'FOODS_1_001_CA_3'
 'FOODS_1_001_CA_4' 'FOODS_1_001_TX_1' 'FOODS_1_001_TX_2'
 'FOODS_1_001_TX_3' 'FOODS_1_001_WI_1' 'FOODS_1_001_WI_2'
 'FOODS_1_001_WI_3']


In [ ]:
cnt = 0
cur = 0
prev = 0
for instance in duplicate_id_list:
  cnt += 1
  if cnt % 1000 == 0:
      print(f"Processing instance {cnt}/{len(duplicate_id_list)}")
      
  cur += 1
  while cur < len(grid_df) and grid_df.iloc[cur]['id'] == instance:
      cur += 1
  filtered_df = grid_df.iloc[prev:cur]
  prev = cur
  
  sales_ts = pd.DataFrame({
     'timestamps': filtered_df['d'].values,
     'demand': filtered_df['demand'].values
  })
  sales_ts['d'] = sales_ts['timestamps'] # keep sales_ts['d'] as integer

  sales_ts['EMA'] = sales_ts['demand'].ewm(span=2, adjust=False).mean()
  sales_ts['EMA'] = sales_ts['EMA'].apply(math.ceil)


  # Phân tích chuỗi theo mùa vụ
  stl = STL(sales_ts['demand'], period=30)  # nếu data theo ngày, 7 là 1 tuần
  res = stl.fit()

  residual = res.resid

  # Áp dụng z-score để tìm outlier trong residual
  z = np.abs(stats.zscore(residual))
  threshold = 3.5  # ngưỡng z-score
  outliers = sales_ts[z > threshold]
  #print(outliers.head())

  step = prev
  for idx, row in outliers.iterrows():
      while row['d'] != grid_df.iloc[step]['d']:
          step += 1
      grid_df.at[step, 'demand'] = row['EMA']


Processing instance 1000/30490
Processing instance 2000/30490
Processing instance 3000/30490
Processing instance 4000/30490
Processing instance 5000/30490
Processing instance 6000/30490
Processing instance 7000/30490
Processing instance 8000/30490
Processing instance 9000/30490
Processing instance 10000/30490
Processing instance 11000/30490
Processing instance 12000/30490
Processing instance 13000/30490
Processing instance 14000/30490
Processing instance 15000/30490
Processing instance 16000/30490
Processing instance 17000/30490
Processing instance 18000/30490
Processing instance 19000/30490
Processing instance 20000/30490
Processing instance 21000/30490
Processing instance 22000/30490
Processing instance 23000/30490
Processing instance 24000/30490
Processing instance 25000/30490
Processing instance 26000/30490
Processing instance 27000/30490
Processing instance 28000/30490
Processing instance 29000/30490
Processing instance 30000/30490


IndexError: single positional indexer is out-of-bounds

In [ ]:
grid_df.to_parquet("done_outlier_data.parquet", engine="pyarrow", index=False)

In [ ]:
print(grid_df.shape)

(27837370, 29)
